In [ ]:
import pandas as pd
import numpy as np
import kagglehub
from sklearn.model_selection import train_test_split
from pathlib import Path

from capstone.gpu import configure_gpu
from capstone.dataset import prepare_experiment
from capstone.classic import train_classical_model, train_classical_model_cached, classical_predict_proba
from capstone.evaluate import evaluate_model
from capstone.dnn import train_dnn, train_dnn_cached

%load_ext autoreload
%autoreload 2


I0000 00:00:1786119902.770119  385069 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786119902.789899  385069 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786119903.274868  385069 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
configure_gpu()


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
tf.Tensor(645.3234, shape=(), dtype=float32)


I0000 00:00:1786119904.471208  385069 gpu_device.cc:2023] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 965 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5090, pci bus id: 0000:01:00.0, compute capability: 12.0a


## TODO

In [9]:
# Download the consoldiated spam data set from kaggle; the kaggle API handles local caching
path = kagglehub.dataset_download("nitishabharathi/email-spam-dataset")
print(path)
raw_data_sets = { 
    'Enron': pd.read_csv(f"{path}/enronSpamSubset.csv"),
    'Spam Assassin': pd.read_csv(f"{path}/completeSpamAssassin.csv"),
    'LingSpam': pd.read_csv(f"{path}/lingSpam.csv")
}

/home/masimms/.cache/kagglehub/datasets/nitishabharathi/email-spam-dataset/versions/1


In [5]:
enron_train, enron_test, enron_features = prepare_experiment(
    { "Enron" : raw_data_sets["Enron"] }, stratify_by_source=False
)
combined_train, combined_test, combined_features = prepare_experiment(
    raw_data_sets, stratify_by_source=True
)

## TODO

In [7]:
%%time
enron_grid = train_classical_model_cached(Path("artifacts/classic_enron.joblib"), enron_train, feature_cols=enron_features, verbose=1)
print("Enron only best parameters: ", enron_grid.best_params_)

Loading cached model from artifacts/classic_enron.joblib
Enron only best parameters:  {'clf__C': 100, 'features__tfidf__max_features': 20000, 'features__tfidf__ngram_range': (1, 1)}
CPU times: user 31.8 ms, sys: 0 ns, total: 31.8 ms
Wall time: 31.5 ms


In [8]:
%%time
combined_grid = train_classical_model_cached(Path("artifacts/classic_combined.joblib"), combined_train, feature_cols=combined_features, verbose=1)
print("Combined best parameters: ", combined_grid.best_params_)

Loading cached model from artifacts/classic_combined.joblib
Combined best parameters:  {'clf__C': 100, 'features__tfidf__max_features': None, 'features__tfidf__ngram_range': (1, 1)}
CPU times: user 85.7 ms, sys: 0 ns, total: 85.7 ms
Wall time: 85.5 ms


### TODO

In [ ]:
df_full = pd.concat([combined_train, combined_test], ignore_index=True)
df_full = df_full[df_full["Source"] != "Enron"]

enron_predict_proba = classical_predict_proba(enron_grid.best_estimator_)
combined_predict_proba = classical_predict_proba(combined_grid.best_estimator_)

results = { 
    "Enron only" : evaluate_model(enron_predict_proba, enron_test, "Label"),
    "Enron only (isolated test data)" : evaluate_model(enron_predict_proba, df_full, "Label"),
    "Combined": evaluate_model(combined_predict_proba, combined_test, "Label")
}

In [ ]:
for source_name, source_df in combined_test.groupby("Source"):
    results[f"Combined ({source_name})"] = evaluate_model(combined_predict_proba, source_df, "Label")

for name, metrics in results.items():
    print(f"{name}: precision={metrics['precision']:.3f} recall={metrics['recall']:.3f} "
          f"f1={metrics['f1']:.3f} roc_auc={metrics['roc_auc']:.3f}")

### DNN

In [ ]:
%%time
enron_dnn = train_dnn_cached(enron_train, enron_features, "artifacts/dnn_enron.keras")

In [ ]:
%%time
combined_dnn = train_dnn_cached(combined_train, combined_features, "artifacts/dnn_combined.keras")

In [ ]:
df_clean = pd.concat([combined_train, combined_test], ignore_index=True)
df_clean = df_clean[df_clean["Source"] != "Enron"]

enron_dnn_predict_proba = dnn_predict_proba(enron_dnn, enron_features)
combined_dnn_predict_proba = dnn_predict_proba(combined_dnn, combined_features)

dnn_results = {
    "Enron-only (in-distribution)": evaluate_model(enron_dnn_predict_proba, enron_test, "Label"),
    "Enron-only (out-of-distribution)": evaluate_model(enron_dnn_predict_proba, df_clean, "Label"),
    "Combined (overall)": evaluate_model(combined_dnn_predict_proba, combined_test, "Label"),
}

for source_name, source_df in combined_test.groupby("Source"):
    dnn_results[f"Combined ({source_name})"] = evaluate_model(combined_dnn_predict_proba, source_df, "Label")

for name, metrics in dnn_results.items():
    print(f"{name}: precision={metrics['precision']:.3f} recall={metrics['recall']:.3f} "
          f"f1={metrics['f1']:.3f} roc_auc={metrics['roc_auc']:.3f}")